# Model 1 — Flat DDM
**Outputs:**
- `models/results/flat_fit.pkl` — serialised CmdStanPy fit object
- `models/results/flat_summary.csv` — posterior summary for all parameters

In [2]:
import pickle
import pandas as pd
from pathlib import Path
from cmdstanpy import CmdStanModel

DATA_PATH = Path("../EDA/data/dataset.csv")
STAN_FILE = Path("flat.stan")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

N_CHAINS = 4
N_WARMUP = 1000
N_SAMPLES = 1000
SEED = 42

assert DATA_PATH.exists(), f"dataset.csv not found at {DATA_PATH}"
assert STAN_FILE.exists(), f"Stan file not found at {STAN_FILE}"

def load_and_validate(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = ["choice", "RT", "difficulty_bin"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df = df.dropna(subset=required)
    df = df[(df["RT"] > 0.05)]

    print(f"Loaded: {len(df):,} trials")
    print(f"RT range: {df['RT'].min():.3f}s -- {df['RT'].max():.3f}s")
    print(f"Choice balance: {df['choice'].mean():.1%} offensive")
    return df

def build_stan_data(df: pd.DataFrame) -> dict:
    return {
        "N": len(df),
        "choice": df["choice"].astype(int).tolist(),
        "RT": df["RT"].tolist(),
        "difficulty": df["difficulty_bin"].astype(int).tolist()
    }

df = load_and_validate(DATA_PATH)
stan_data = build_stan_data(df)

model = CmdStanModel(stan_file=str(STAN_FILE))
fit = model.sample(
    data=stan_data,
    chains=N_CHAINS,
    iter_warmup=N_WARMUP,
    iter_sampling=N_SAMPLES,
    seed=SEED,
    show_progress=True
)

summary = fit.summary()
summary.to_csv(RESULTS_DIR / "flat_summary.csv")

with open(RESULTS_DIR / "flat_fit.pkl", "wb") as f:
    pickle.dump(fit, f)

summary.loc[["v_easy", "v_hard", "v_diff", "beta", "p_v_easy_gt_v_hard"]]

/Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-Capstone-Project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
21:09:42 - cmdstanpy - INFO - compiling stan file /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-Capstone-Project/models/flat.stan to exe file /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-Capstone-Project/models/flat


Loaded: 22,263 trials
RT range: 0.100s -- 2.933s
Choice balance: 60.3% offensive


21:09:46 - cmdstanpy - INFO - compiled model executable: /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-Capstone-Project/models/flat
21:09:46 - cmdstanpy - INFO - CmdStan start processing
chain 2:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]


chain 1:   0%|          | 1/2000 [00:00<17:09,  1.94it/s, (Warmup)]A


chain 2:   0%|          | 1/2000 [00:00<17:30,  1.90it/s, (Warmup)]

chain 3:   0%|          | 1/2000 [00:00<18:48,  1.77it/s, (Warmup)]

chain 3:   5%|▌         | 100/2000 [00:14<04:40,  6.78it/s, (Warmup)]


chain 2:   5%|▌         | 100/2000 [00:17<05:21,  5.92it/s, (Warmup)]

chain 3:  10%|█         | 200/2000 [00:32<04:53,  6.13it/s, (Warmup)]


chain 2:  10%|█         | 200/2000 [00:35<05:22,  5.58it/s, (Warmup)]

chain 1:  15%|█▌        | 300/2000 [00:50<04:52,  5.82it/s, (Warmup)]


chain 2:  15%|█▌        | 300/2000 [00:52<05:00,  5.66it/s, (Warmup)]

chain 1:  20%|██        | 400/2000 [01:03


21:14:26 - cmdstanpy - INFO - CmdStan done processing.


,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
v_easy,0.303973,0.000264,0.012921,0.012811,0.282203,0.303836,0.325314,2430.93,2733.58,4.51868,1.001090
v_hard,0.286643,0.000212,0.010634,0.010782,0.269091,0.286733,0.303918,2568.63,3013.12,4.77464,1.000260
v_diff,0.017330,0.000258,0.015064,0.015008,-0.007157,0.017273,0.042353,3422.73,2964.81,6.36226,1.000200
beta,0.452066,0.000053,0.002520,0.002497,0.447876,0.452109,0.456113,2296.67,2444.91,4.26911,1.001520
p_v_easy_gt_v_hard,0.874750,NaN,0.331044,0.000000,0.000000,1.000000,1.000000,3360.61,3360.61,6.24679,0.999651


## Convergence Check

In [3]:
def check_convergence(fit) -> pd.DataFrame:
    """
    Check R-hat and ESS for all parameters.
    Flags any parameter that fails the threshold.
    Returns the full summary dataframe.
    """
    summary  = fit.summary()
    diagnose = fit.diagnose()

    # Divergences
    print(diagnose)

    # R-hat
    rhat_col = [c for c in summary.columns if "r_hat" in c.lower()][0]
    bad_rhat = summary[summary[rhat_col] >= 1.01]
    if bad_rhat.empty:
        print(f"All R-hat < 1.01 ({len(summary)} parameters checked)")
    else:
        print(f"{len(bad_rhat)} parameters with R-hat >= 1.01:")
        print(bad_rhat[[rhat_col]].to_string())

    # ESS
    ess_col = [c for c in summary.columns if "ess_bulk" in c.lower()][0]
    bad_ess = summary[summary[ess_col] < 400]
    if bad_ess.empty:
        print("All ESS > 400")
    else:
        print(f"{len(bad_ess)} parameters with ESS < 400:")
        print(bad_ess[[ess_col]].to_string())

    return summary


summary = check_convergence(fit)

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete, no problems detected.

All R-hat < 1.01 (22270 parameters checked)
All ESS > 400
